In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize, LinearSegmentedColormap

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
FLOW_ROOT = ANALYSIS_ROOT / 'beta_effect_background_flow'
ELLIPSE_ROOT = ANALYSIS_ROOT / 'ellipse_tilt_analysis'
for path in (ANALYSIS_ROOT, CASE_ROOT, FLOW_ROOT, ELLIPSE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, load_background_cache
import ellipse_tilt_tools as et

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 60)

In [2]:
ELLIPSE_FRAC = 1.0

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_pv_gradient_terms(
    df_eddies,
    grid,
    core_mean=True,
    frac=ELLIPSE_FRAC,
    surface_method="esp_gaussian",
    averaging="nonlinear",
)
df_eddies = tilt.add_region_labels(df_eddies, grid)

# Surface ellipse geometry and axial tilt alignment (0° parallel, 90° perpendicular).
df_eddies = et.ellipse_geometry(df_eddies)
df_eddies['TiltMajorAlignmentDeg'] = np.abs(
    et.axial_difference(df_eddies.TiltDir, df_eddies.MajorBearing)
)

# Primary environmental shear used in the background-flow analysis:
# centred seasonal climatology, surface minus the 200–500 m layer mean.
background = load_background_cache(BackgroundConfig()).drop(
    columns=['ic', 'jc'], errors='ignore'
)
needed = {'clim_surface_east_ms', 'clim_surface_north_ms',
          'clim_200_east_ms', 'clim_200_north_ms',
          'clim_500_east_ms', 'clim_500_north_ms'}
missing = needed - set(background.columns)
if missing:
    raise RuntimeError(f'Background cache is missing {sorted(missing)}')
df_eddies = df_eddies.merge(
    background[['Eddy', 'Day', *sorted(needed)]],
    on=['Eddy', 'Day'], how='left', validate='one_to_one'
)
for component in ('east', 'north'):
    deep = (500 * df_eddies[f'clim_500_{component}_ms']
            - 200 * df_eddies[f'clim_200_{component}_ms']) / 300
    df_eddies[f'clim_upper_minus_deep_{component}_ms'] = (
        df_eddies[f'clim_200_{component}_ms'] - deep
    )
SHEAR_E = 'clim_upper_minus_deep_east_ms'
SHEAR_N = 'clim_upper_minus_deep_north_ms'

In [5]:
df_eddies.columns


Index(['Eddy', 'Day', 'Cyc', 'lon', 'lat', 'ic', 'jc', 'xc', 'yc', 'w',
       'Omega', 'q11', 'q12', 'q22', 'Rc', 'psi0', 'AR', 'R', 'Age', 'Date',
       'fname', 'TiltDis', 'TiltDir', 'h', 'f', 'beta', 'dhdx', 'dhdy',
       'zeta_mean', 'abs_vort', 'PV', 'PV_footprint_n',
       'PV_weight_effective_n', 'PV_grad_plan_x', 'PV_grad_plan_y',
       'PV_grad_plan_mean_local_mag', 'PV_grad_plan_rms_local_mag',
       'PV_grad_plan_p90_local_mag', 'PV_grad_plan_coherence',
       'PV_grad_topo_x', 'PV_grad_topo_y', 'PV_grad_topo_mean_local_mag',
       'PV_grad_topo_rms_local_mag', 'PV_grad_topo_p90_local_mag',
       'PV_grad_topo_coherence', 'PV_grad_x', 'PV_grad_y',
       'PV_grad_mean_local_mag', 'PV_grad_rms_local_mag',
       'PV_grad_p90_local_mag', 'PV_grad_coherence', 'PV_grad_eddy_x',
       'PV_grad_eddy_y', 'PV_grad_eddy_mean_local_mag',
       'PV_grad_eddy_rms_local_mag', 'PV_grad_eddy_p90_local_mag',
       'PV_grad_eddy_coherence', 'PV_grad_full_x', 'PV_grad_full_y',
   